# Next-Concept Modeling (NCM): a tiny, runnable walkthrough

A toy re-implementation of the core ideas of *Next-Concept Modeling: Fast, Parallel Coarse-to-Fine Text Generation* (Kumbong et al., 2026), small enough to train in a few minutes on a free Colab GPU (or CPU).

**The idea in one picture**

```
Autoregressive : t1 -> t2 -> t3 -> ... -> t48              (48 forward passes, one token each)
Diffusion      : noise -> ... -> ... -> text               (many denoising passes)
NCM            : [3 codes] -> [6] -> [12] -> [24] -> [48]  (5 forward passes, each level fully parallel)
```

Two stages:

1. **Tokenizer (multiscale residual VQ-VAE).** A sequence is encoded to vectors `z`. We quantize it in several rounds: level 0 captures a very coarse summary with only 3 codes, level 1 quantizes the *residual* with 6 codes, and so on until the full resolution. Summing the (upsampled) quantized levels reconstructs the text, more precisely at each level.
2. **Generator (next-concept transformer).** Predicts all codes of level *k* **in parallel**, given the "concept" (the partial reconstruction) from levels `< k`.

This notebook is self-contained (no downloads). Real NCM uses far bigger models and real datasets, see the repo for that.

In [ ]:
import math, random, time
import torch, torch.nn as nn, torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); random.seed(0)
print("device:", device)

# Lower these if you just want a fast smoke test.
VQ_STEPS  = 2500   # tokenizer training steps
GEN_STEPS = 3000   # generator training steps

## 1. Toy data

A tiny templated "language" (`the <adj> <noun> <verb> <prep> the <place>.`), padded to 48 characters. Character-level, so the tokenizer has to *discover* words and structure by itself.

In [ ]:
adjs   = ["big", "small", "red", "old", "happy", "tiny"]
nouns  = ["cat", "dog", "bird", "boy", "girl", "fox"]
verbs  = ["sat", "ran", "slept", "jumped", "played"]
preps  = ["on", "near", "under", "by"]
places = ["mat", "tree", "house", "box", "hill"]

T, PAD = 48, "~"
def make_sentence():
    s = f"the {random.choice(adjs)} {random.choice(nouns)} {random.choice(verbs)} {random.choice(preps)} the {random.choice(places)}."
    return s.ljust(T, PAD)

chars = sorted(set("abcdefghijklmnopqrstuvwxyz .") | {PAD})
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
V_CHAR = len(chars)

encode_text = lambda s: torch.tensor([stoi[c] for c in s])
decode_ids  = lambda ids: "".join(itos[int(i)] for i in ids)

data = torch.stack([encode_text(make_sentence()) for _ in range(20000)]).to(device)
print(data.shape, "|", decode_ids(data[0]), "|")

## 2. The multiscale residual vector quantizer

This is the heart of the tokenizer (cf. `MultiscaleResidualVectorQuantizer` in `models/quant.py`). For each level `i` with `levels[i]` codes:

1. **Downsample** the *residual* (what previous levels haven't explained yet) to `levels[i]` positions (area pooling).
2. **Quantize**: nearest codebook vector per position -> discrete codes.
3. **Upsample** the quantized vectors back to full length (linear interpolation), **add** to the running reconstruction `z_hat`, and subtract from the residual.

Codebooks are learned by exponential moving average (no gradients), as in the repo.

**Fine-level dropout** (an option from the repo, off by default here): with `fine_drop > 0`, that fraction of the time the decoder only receives the reconstruction from a random *truncated* set of levels. It pushes coarse levels to carry more meaning, but needs longer training to pay off (try `fine_drop=0.2` with more steps). In this toy the finest level does most of the work.

In [ ]:
LEVELS = [3, 6, 12, 24, 48]     # codes per level (the last one equals the sequence length)
CODES  = 64                     # codebook size per level
D_VQ   = 64                     # embedding dim of the tokenizer

def down(z, l):   # (B,T,D) -> (B,l,D)   area-pool
    return z if z.shape[1] == l else F.interpolate(z.transpose(1, 2), size=l, mode="area").transpose(1, 2)

def up(q, L):     # (B,l,D) -> (B,L,D)   linear upsample
    return q if q.shape[1] == L else F.interpolate(q.transpose(1, 2), size=L, mode="linear").transpose(1, 2)

class MultiscaleRVQ(nn.Module):
    def __init__(self, levels, codes, dim, decay=0.99, eps=1e-5, fine_drop=0.0):
        super().__init__()
        self.fine_drop = fine_drop
        self.levels, self.codes, self.dim, self.decay, self.eps = levels, codes, dim, decay, eps
        n = len(levels)
        self.register_buffer("cb",    torch.randn(n, codes, dim))
        self.register_buffer("ema_n", torch.ones(n, codes))
        self.register_buffer("ema_w", self.cb.clone())
        self.inited = [False] * n

    @torch.no_grad()
    def _ema_update(self, i, flat, idx):
        if not self.inited[i]:                                  # init codebook from data
            self.cb[i] = flat[torch.randint(len(flat), (self.codes,), device=flat.device)]
            self.ema_w[i] = self.cb[i]; self.inited[i] = True
        onehot = F.one_hot(idx, self.codes).float()
        self.ema_n[i].mul_(self.decay).add_(onehot.sum(0), alpha=1 - self.decay)
        self.ema_w[i].mul_(self.decay).add_(onehot.t() @ flat, alpha=1 - self.decay)
        n = self.ema_n[i].sum()
        smooth = (self.ema_n[i] + self.eps) / (n + self.codes * self.eps) * n
        self.cb[i] = self.ema_w[i] / smooth[:, None]
        dead = self.ema_n[i] < 0.03                              # restart unused codes
        if dead.any():
            new = flat[torch.randint(len(flat), (int(dead.sum()),), device=flat.device)]
            self.cb[i][dead] = new; self.ema_w[i][dead] = new; self.ema_n[i][dead] = 1.0

    def forward(self, z):
        B, L, D = z.shape
        z_rest, z_hat, idx_list, partial = z.detach(), torch.zeros_like(z), [], []
        for i, l in enumerate(self.levels):
            flat = down(z_rest, l).reshape(-1, D)
            dist = flat.pow(2).sum(1, keepdim=True) - 2 * flat @ self.cb[i].t() + self.cb[i].pow(2).sum(1)
            idx = dist.argmin(1)
            if self.training:
                self._ema_update(i, flat, idx)
            q = up(self.cb[i][idx].view(B, l, D), L)
            z_hat, z_rest = z_hat + q, z_rest - q
            idx_list.append(idx.view(B, l))
            partial.append(z_hat)
        commit = F.mse_loss(z, z_hat.detach())
        if self.training and random.random() < self.fine_drop:   # fine-level dropout (see text above)
            z_hat = partial[random.randint(0, len(partial) - 2)]
        z_st = z + (z_hat - z).detach()                          # straight-through
        return z_st, commit, idx_list

    # ---- used at generation time -------------------------------------------------
    def add_level(self, z_hat, idx, i):
        # z_hat += (upsampled) codebook vectors of level i
        return z_hat + up(self.cb[i][idx], self.levels[-1])

    def idx_to_zhats(self, idx_list):
        z_hat = torch.zeros(idx_list[0].shape[0], self.levels[-1], self.dim, device=self.cb.device)
        out = []
        for i, idx in enumerate(idx_list):
            z_hat = self.add_level(z_hat, idx, i); out.append(z_hat)
        return out                                               # cumulative reconstruction after each level

## 3. The text VQ-VAE (encoder -> multiscale RVQ -> decoder)

In [ ]:
def stack(d, layers, heads=4):
    layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.0, activation="gelu",
                                       batch_first=True, norm_first=True)
    return nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)

class TextVQVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V_CHAR, D_VQ)
        self.pos_e, self.pos_d = nn.Embedding(T, D_VQ), nn.Embedding(T, D_VQ)
        self.enc, self.dec = stack(D_VQ, 2), stack(D_VQ, 2)
        self.out = nn.Linear(D_VQ, V_CHAR)
        self.rvq = MultiscaleRVQ(LEVELS, CODES, D_VQ)

    def encode_z(self, x):   return self.enc(self.emb(x) + self.pos_e.weight[: x.shape[1]])
    def decode_z(self, z):   return self.out(self.dec(z + self.pos_d.weight[: z.shape[1]]))

    def forward(self, x):
        z = self.encode_z(x)
        z_st, commit, idx_list = self.rvq(z)
        logits = self.decode_z(z_st)
        rec = F.cross_entropy(logits.reshape(-1, V_CHAR), x.reshape(-1))
        return rec + 0.25 * commit, rec, logits, idx_list

vq = TextVQVAE().to(device)
opt = torch.optim.AdamW(vq.parameters(), lr=2e-3)
t0 = time.time()
for step in range(1, VQ_STEPS + 1):
    x = data[torch.randint(len(data), (128,), device=device)]
    loss, rec, logits, _ = vq(x)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 250 == 0 or step == 1:
        acc = (logits.argmax(-1) == x).float().mean().item()
        print(f"step {step:5d} | loss {loss.item():.4f} | recon CE {rec.item():.4f} | char acc {acc:.3f} | {time.time()-t0:.0f}s")

### Look at the hierarchy: coarse -> fine reconstruction

Take a sentence, encode it into codes, and decode after only the first level, then the first two, and so on. Early levels should capture the rough content, later levels fix details.

In [ ]:
vq.eval()
@torch.no_grad()
def show_levels(x):
    z = vq.encode_z(x)
    _, _, idx_list = vq.rvq(z)
    for k, zh in enumerate(vq.rvq.idx_to_zhats(idx_list)):
        txt = decode_ids(vq.decode_z(zh).argmax(-1)[0])
        print(f"levels 0..{k} ({sum(LEVELS[:k+1]):3d} codes) | {txt} |")
    print(f"original               | {decode_ids(x[0])} |")
    return idx_list

idx_list = show_levels(data[:1])
print("\nlevel-0 codes:", idx_list[0][0].tolist(), "(3 numbers summarise the whole sentence)")

# average character accuracy using only the first k levels
with torch.no_grad():
    x = data[:2000]
    _, _, il = vq.rvq(vq.encode_z(x))
    for k, zh in enumerate(vq.rvq.idx_to_zhats(il)):
        print(f"levels 0..{k}: char accuracy {(vq.decode_z(zh).argmax(-1) == x).float().mean():.3f}")

## 4. Pre-tokenize the dataset

The tokenizer is now **frozen**. Every sentence becomes a list of code tensors, one per level (3 + 6 + 12 + 24 + 48 = 93 codes in total).

In [ ]:
@torch.no_grad()
def tokenize(x):
    return vq.rvq(vq.encode_z(x))[2]

all_idx = [[] for _ in LEVELS]
for chunk in data.split(1000):
    for k, idx in enumerate(tokenize(chunk)):
        all_idx[k].append(idx)
all_idx = [torch.cat(a) for a in all_idx]
print([tuple(a.shape) for a in all_idx])

## 5. The next-concept generator

The input to level *k* is the **concept** so far: the running reconstruction `z_hat` from levels `< k`, pooled down to `LEVELS[k]` positions. Level 0 has nothing to condition on, so it gets learned query vectors.

All levels are packed into one sequence with a **block-diagonal attention mask**: tokens only attend within their own level. This works because each level's input already summarises everything coarser. So at training time all levels are learned in one forward pass (teacher forcing), and at inference each level needs exactly **one** pass, with every position in the level sampled in parallel.

In [ ]:
D_GEN = 128

def sample(logits, temperature=1.0, top_k=0):
    if temperature == 0:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = logits.topk(top_k, -1).values[..., -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    p = F.softmax(logits, -1)
    return torch.multinomial(p.reshape(-1, p.shape[-1]), 1).view(p.shape[:-1])

class NextConceptModel(nn.Module):
    def __init__(self, rvq):
        super().__init__()
        self.rvq, self.levels = rvq, rvq.levels
        self.inp  = nn.Linear(rvq.dim, D_GEN)
        self.pos  = nn.Embedding(max(self.levels), D_GEN)      # positions restart at 0 for each level
        self.lvl  = nn.Embedding(len(self.levels), D_GEN)      # which level am I in?
        self.q0   = nn.Parameter(torch.randn(1, self.levels[0], rvq.dim) * 0.02)   # level-0 queries
        self.body = stack(D_GEN, 4)
        self.ln   = nn.LayerNorm(D_GEN)
        self.heads = nn.ModuleList([nn.Linear(D_GEN, rvq.codes) for _ in self.levels])

        lvl_ids = torch.cat([torch.full((l,), i) for i, l in enumerate(self.levels)])
        self.register_buffer("pos_ids", torch.cat([torch.arange(l) for l in self.levels]))
        self.register_buffer("lvl_ids", lvl_ids)
        self.register_buffer("block_mask", lvl_ids[:, None] != lvl_ids[None, :])   # True = not allowed to attend

    def _embed(self, x, pos_ids, lvl_ids):
        return self.inp(x) + self.pos(pos_ids) + self.lvl(lvl_ids)

    def _heads(self, h, lvl_ids):
        return [self.heads[k](h[:, (lvl_ids == k).nonzero().flatten()]) for k in range(len(self.levels))]

    # ---- training: all levels at once, block-diagonal attention -------------------
    def concept_inputs(self, idx_list):
        zhats = self.rvq.idx_to_zhats(idx_list)
        B = idx_list[0].shape[0]
        return [self.q0.expand(B, -1, -1)] + [down(zhats[k - 1], self.levels[k]) for k in range(1, len(self.levels))]

    def forward(self, idx_list):
        x = torch.cat(self.concept_inputs(idx_list), dim=1)
        h = self.body(self._embed(x, self.pos_ids, self.lvl_ids), mask=self.block_mask)
        return self._heads(self.ln(h), self.lvl_ids)         # list of (B, l_k, CODES)

    # ---- inference: ONE forward pass per level ---------------------------------------
    def forward_level(self, x, k):
        l = x.shape[1]
        pos = torch.arange(l, device=x.device)
        h = self.body(self._embed(x, pos, torch.full_like(pos, k)))
        return self.heads[k](self.ln(h))

    @torch.no_grad()
    def generate(self, B, temperature=1.0, top_k=0):
        idx_list, z_hat = [], torch.zeros(B, self.levels[-1], self.rvq.dim, device=self.q0.device)
        x = self.q0.expand(B, -1, -1)
        for k in range(len(self.levels)):
            idx = sample(self.forward_level(x, k), temperature, top_k)     # all positions of level k in parallel
            idx_list.append(idx)
            z_hat = self.rvq.add_level(z_hat, idx, k)
            if k + 1 < len(self.levels):
                x = down(z_hat, self.levels[k + 1])                        # the next "concept"
        return idx_list

gen = NextConceptModel(vq.rvq).to(device)
print(f"generator params: {sum(p.numel() for p in gen.parameters())/1e6:.2f}M")

In [ ]:
# Sanity check: with the block-diagonal mask, a level never sees other levels,
# so running a level alone gives the same logits as the packed training pass.
gen.eval()
with torch.no_grad():
    idx_b = [a[:4] for a in all_idx]
    packed = gen(idx_b)
    xs = gen.concept_inputs(idx_b)
    for k in range(len(LEVELS)):
        alone = gen.forward_level(xs[k], k)
        print(f"level {k}: max |packed - alone| = {(packed[k] - alone).abs().max():.2e}")

### Train the generator

In [ ]:
gen.train()
opt = torch.optim.AdamW(gen.parameters(), lr=1e-3, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=GEN_STEPS, pct_start=0.05)
t0 = time.time()
for step in range(1, GEN_STEPS + 1):
    b = torch.randint(len(data), (128,), device=device)
    idx_b = [a[b] for a in all_idx]
    logits = gen(idx_b)
    losses = [F.cross_entropy(lg.reshape(-1, CODES), tg.reshape(-1)) for lg, tg in zip(logits, idx_b)]
    loss = sum(losses) / len(losses)
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
    opt.step(); sched.step()
    if step % 250 == 0 or step == 1:
        print(f"step {step:5d} | loss {loss.item():.3f} | per level: " + " ".join(f"{l.item():.2f}" for l in losses) + f" | {time.time()-t0:.0f}s")

## 6. Generate text, coarse to fine

Each sample takes just **5 forward passes** (one per level). We decode the partial reconstruction after each level so you can watch the sentence get refined.

In [ ]:
gen.eval(); vq.eval()
torch.manual_seed(1)
with torch.no_grad():
    idx_list = gen.generate(4, temperature=0.8, top_k=10)
    zhats = vq.rvq.idx_to_zhats(idx_list)
    for b in range(4):
        print(f"--- sample {b} ---")
        for k, zh in enumerate(zhats):
            print(f"after level {k} ({LEVELS[k]:2d} codes): {decode_ids(vq.decode_z(zh[b:b+1]).argmax(-1)[0])}")

In [ ]:
# How often does the model produce a sentence from the true grammar?
def valid(s):
    s = s.rstrip(PAD)
    w = s.rstrip(".").split(" ")
    return (s.endswith(".") and len(w) == 7 and w[0] == "the" and w[1] in adjs and w[2] in nouns and w[3] in verbs
            and w[4] in preps and w[5] == "the" and w[6] in places)

with torch.no_grad():
    N = 512
    zh = vq.rvq.idx_to_zhats(gen.generate(N, temperature=0.8, top_k=10))[-1]
    texts = [decode_ids(r) for r in vq.decode_z(zh).argmax(-1)]
print(f"{sum(map(valid, texts))}/{N} samples follow the grammar exactly; {len(set(texts))} unique")

# Speed: 5 parallel level-passes vs. one pass per character for an autoregressive model.
def bench(f, n=5):
    if device == "cuda": torch.cuda.synchronize()
    t = time.time(); [f() for _ in range(n)]
    if device == "cuda": torch.cuda.synchronize()
    return (time.time() - t) / n
t_ncm = bench(lambda: gen.generate(64, 0.8, 10))
print(f"NCM: {len(LEVELS)} forward passes / sample, {t_ncm*1000:.1f} ms for 64 samples")
print(f"A char-level autoregressive model would need {T} forward passes for the same {T} characters.")

## What to explore next

* **Temperature / top-k:** change them in `generate` and see diversity vs. correctness.
* **Fewer levels:** set `LEVELS = [6, 12, 24, 48]` (or `[4, 16, 48]`) and compare quality vs. number of passes. This is the quality/speed knob in the paper.
* **Codebook size / dimension:** shrink `CODES` and watch reconstruction at coarse levels degrade.
* **Conditioning:** the paper's XSum/WMT14 tasks add a *prefix* (the source document or sentence) that every level can attend to. Here that would mean prepending encoded context tokens to the sequence and letting all levels attend to them.
* **Extra tricks from the repo** (not in this toy): classifier-free guidance, fine-level dropout during tokenizer training, learned up/down-samplers, a residual conv after upsampling (`Phi`), KV caching of the prefix, and tying output heads to the codebook.

**Mapping to the repo:** `MultiscaleRVQ` ~ `models/quant.py`, `TextVQVAE` ~ `models/vqvae.py` + `vae.py`, `NextConceptModel` ~ `models/next_concept.py` (`attn_pattern='block_diagonal'`, `query_level0`, `generate`).